# GDG on Campus HWU
### A Retrieval-Augmented Generation (RAG) Starter Notebook

Welcome! By the end of **Session 1**, you'll have built a working search engine over real text — first on a shared example, then on a topic of *your own choosing*.

**Today we are only building retrieval (search).** Generation (turning retrieved text into a spoken answer) is Session 2.

Run the cells top to bottom. Don't skip the setup cells — they install and configure everything we need.


## Step 0 — Setup
Run this first. It installs everything we need for today.

In [1]:
!pip install -q google-genai numpy wikipedia pypdf requests
print("✅ Packages installed.")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 15.0 MB/s eta 0:00:00
✅ Packages installed.


### Get your free Gemini API key

You should have already created this **before today** (see the pre-session email). If not:
1. Go to https://aistudio.google.com/apikey
2. Sign in with your Google account
3. Click "Create API key" and copy it

Paste it below when prompted. It stays private to your own Colab session — we never see it, and it's never written to this notebook file.

In [3]:
from getpass import getpass

API_KEY = getpass("Paste your Gemini API key here and press Enter: ")

from google import genai
client = genai.Client(api_key=API_KEY)

print("✅ Client ready.")


Paste your Gemini API key here and press Enter: ··········
✅ Client ready.


---
## PART 1 — Learn the pipeline on a shared example

We'll build **chunk → embed → retrieve** together, step by step, using one shared demo topic: the **Burj Khalifa**.

Once this works, Part 2 asks you to point the exact same code at a topic *you* picked.


### Step 1 — The Universal Loader

RAG doesn't care *what* your text is about — it only cares that you can get it into one plain string. This function can pull text from four kinds of sources:

| `source_type` | `source` should be... | Notes |
|---|---|---|
| `"wikipedia"` | an exact article title, e.g. `"Burj Khalifa"` | Cleanest option — use this when you can |
| `"gdoc"` | a Google Doc link or ID | Doc must be shared as "Anyone with the link can view" |
| `"pdf"` | a file path to an uploaded `.pdf` | Text-based PDFs only — **not** scanned/image PDFs |
| `"txt"` | a file path to an uploaded `.txt` | Always the safest fallback |
| `"paste"` | raw text you paste directly as a string | For anything copy-pasted by hand (e.g. a niche wiki page) |

It also automatically checks the size of what you've loaded and trims it if it's too big — this keeps every team comfortably inside the free API tier's rate limits, regardless of topic.

In [7]:
import re
import requests

MIN_CHARS = 1500
MAX_CHARS = 150000


def _extract_gdoc_id(url_or_id):
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", url_or_id)
    return match.group(1) if match else url_or_id


def _load_wikipedia(title):
    import wikipedia
    wikipedia.set_lang("en")
    page = wikipedia.page(title, auto_suggest=False)
    return page.content


def _load_gdoc(url_or_id):
    doc_id = _extract_gdoc_id(url_or_id)
    export_url = f"https://docs.google.com/document/d/{doc_id}/export?format=txt"
    resp = requests.get(export_url)
    resp.raise_for_status()
    return resp.text


def _load_pdf(path):
    from pypdf import PdfReader
    reader = PdfReader(path)
    pages_text = [page.extract_text() or "" for page in reader.pages]
    text = "\n".join(pages_text)
    if not text.strip():
        raise ValueError(
            "No text could be extracted. This is likely a scanned/image PDF, "
            "which this workshop doesn't support — pick a text-based PDF instead."
        )
    return text


def _load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_text(source_type, source):
    """Universal loader: returns clean plain text regardless of source type,
    and automatically enforces a safe size range for the free API tier."""
    loaders = {
        "wikipedia": _load_wikipedia,
        "gdoc": _load_gdoc,
        "pdf": _load_pdf,
        "txt": _load_txt,
        "paste": lambda s: s,
    }
    if source_type not in loaders:
        raise ValueError(f"Unknown source_type '{source_type}'. Choose one of: {list(loaders.keys())}")

    text = loaders[source_type](source).strip()
    char_count = len(text)
    print(f"Loaded {char_count:,} characters.")

    if char_count < MIN_CHARS:
        print(
            f"⚠️  That's quite short (under {MIN_CHARS:,} chars). Retrieval needs a bit more "
            f"material to actually search over — consider a longer article or section."
        )

    if char_count > MAX_CHARS:
        text = text[:MAX_CHARS]
        print(
            f"⚠️  Trimmed from {char_count:,} to {MAX_CHARS:,} characters to stay safely inside "
            f"the free API tier's rate limits. Tip: pick ONE focused article/section, not an "
            f"entire wiki or book."
        )

    return text


### Step 2 — Load the shared demo dataset

In [26]:
demo_text = load_text("wikipedia", "Burj Khalifa")
print(demo_text[:500], "...")


Loaded 30,263 characters.
The Burj Khalifa (previously known as Burj Dubai prior to inauguration) is a megatall skyscraper in Dubai, United Arab Emirates. Designed by the SOM architectural firm, it is the world's tallest structure, with a total height of 829.8 m (2,722 ft, or just over half a mile) and a roof height (excluding the antenna, but including a 242.6 m spire) of 828 m (2,717 ft). It has been the tallest building in the world since its complete inauguration in 2011, surpassing Taipei 101, which had held the rec ...


---
### Step 3 — Chunking

Why chunk? Two reasons: the model can only usefully focus on a limited amount of text at once and **smaller, well-sized chunks make retrieval more precise** if a chunk is too big, you'll retrieve a lot of irrelevant text along with the relevant sentence.

We're using a simple **fixed-size chunker with overlap**: cut every `chunk_size` characters but let consecutive chunks overlap slightly so we don't accidentally slice a sentence in half at a chunk boundary.

In [44]:
def chunk_text(text, chunk_size=500, overlap=50):
    """
    TODO: Split the text into overlapping chunks.
    - Each chunk should be roughly `chunk_size` characters long.
    - Consecutive chunks should overlap by `overlap` characters to preserve context across boundaries.
    """
    chunks = []

    # --- WRITE YOUR CHUNKING LOGIC HERE ---
    # Hint: Use a while loop with a moving start pointer:
    # start += chunk_size - overlap

    return chunks


# demo_chunks = chunk_text(demo_text)
# print(f"Split into {len(demo_chunks)} chunks.")

---
### Step 4 — Embedding

An embedding turns text into a list of numbers that captures *meaning* — texts with similar meaning end up with similar numbers. We're using Google's `gemini-embedding-001` model.

Gemini allows up to **100 texts per request**, so we batch at the real maximum rather than sending one-at-a-time — this is the single biggest lever on how long embedding takes. We also **checkpoint progress to a local file** as we go: if a cell gets re-run after an error or a typo fix, it picks up where it left off instead of re-embedding everything (and re-spending API calls) from scratch.

In [45]:
import time
import pickle
import os
import numpy as np

MAX_BATCH = 100  # Gemini's actual per-request limit for embed_content


def embed_texts(client, texts, checkpoint_path=None, delay_seconds=1):
    """
    TODO: Embed a list of text chunks using Gemini in batches of up to 100.
    - Handle API rate limits / errors with a retry mechanism.
    - Save progress using checkpoints if `checkpoint_path` is provided.
    """
    all_embeddings = []

    # --- WRITE YOUR BATCHING & EMBEDDING LOGIC HERE ---
    # Hint:
    # result = client.models.embed_content(model="gemini-embedding-001", contents=batch)
    # all_embeddings.extend([e.values for e in result.embeddings])

    return np.array(all_embeddings)


# demo_embeddings = embed_texts(client, demo_chunks, checkpoint_path="demo_checkpoint.pkl")

---
### Step 5 — Retrieval

Given a question, we embed it the same way, then find which chunks are "closest" in meaning using **cosine similarity** a measure of how aligned two vectors are.

Highest similarity = most relevant chunk.
This is the actual mechanism underneath every vector database, we're just doing it by hand in plain NumPy, so you can see exactly what's happening.

In [46]:
def cosine_similarity(a, b):
    """
    TODO: Calculate the cosine similarity between two vectors `a` and `b`.
    Formula: dot product divided by the product of their norms.
    """
    # --- WRITE YOUR MATH HERE ---
    pass


def retrieve(client, query, chunks, chunk_embeddings, top_k=3):
    """
    TODO:
    1. Embed the user's query string using Gemini.
    2. Compute the cosine similarity between the query embedding and every chunk embedding.
    3. Sort the chunks by similarity score in descending order.
    4. Return the top_k chunks and their scores.
    """
    # --- WRITE YOUR RETRIEVAL LOGIC HERE ---
    pass


# results = retrieve(client, "How tall is the Burj Khalifa?", demo_chunks, demo_embeddings, top_k=3)

In [48]:
# Try it! Ask a question about the Burj Khalifa.
query = "How tall is the Burj Khalifa?"

results = retrieve(client, query, demo_chunks, demo_embeddings, top_k=3)

for rank, (chunk, score) in enumerate(results, start=1):
    print(f"#{rank}  (similarity: {score:.3f})")
    print(chunk)
    print("---")


NameError: name 'demo_chunks' is not defined

**Try breaking it.** Ask a question that *isn't* covered in the article (e.g. about a completely different building) and look at the similarity scores — low scores across the board are the system honestly telling you "I don't have a good answer for this." That's a real and important signal, not a bug.

In [ ]:
query = "Try your own question here"

results = retrieve(client, query, demo_chunks, demo_embeddings, top_k=3)
for rank, (chunk, score) in enumerate(results, start=1):
    print(f"#{rank}  (similarity: {score:.3f})")
    print(chunk)
    print("---")


---
## 🎉 Checkpoint

You just built the full **chunk → embed → retrieve** pipeline the "search" half of RAG — from scratch.
Next session we'll connect this to an AI model so it can actually *answer* using what we retrieve instead of just returning raw chunks.

**Part 2 is where it gets personal: same pipeline but make it your topic.**


---
## PART 2 — Build it on YOUR topic

Pick one of the four source types from Step 1, fill in the two variables below and re-run the exact same pipeline.

**Reminder:** one focused article/section, not an entire wiki or book. The loader will warn you (and auto-trim) if you go over but a
tighter topic = better results and faster embedding.

In [ ]:
# ---- EDIT THESE TWO LINES ----
MY_SOURCE_TYPE = "wikipedia"        # one of: "wikipedia", "gdoc", "pdf", "txt", "paste"
MY_SOURCE = "Your topic here"       # article title / gdoc link / file path / pasted text
# -------------------------------

# my_text = load_text(MY_SOURCE_TYPE, MY_SOURCE)
# my_chunks = chunk_text(my_text)

In [ ]:
# my_embeddings = embed_texts(client, my_chunks, checkpoint_path="my_checkpoint.pkl")

In [ ]:
# Ask it something!
my_query = "Ask your bot a question about your own topic"

results = retrieve(client, my_query, my_chunks, my_embeddings, top_k=3)
for rank, (chunk, score) in enumerate(results, start=1):
    print(f"#{rank}  (similarity: {score:.3f})")
    print(chunk)
    print("---")


### Before you go

Try **3 questions** on your own topic:
1. One you're confident it should nail
2. One that's a bit tricky/ambiguous
3. One that's completely outside the topic (to see the "I don't know" signal in action)

And with that we come to the end of Session 1 where we conjigured the 'search' part of building your RAG.

Session 2, we will turn this raw retrieval into an actual spoken answer and look at how to make retrieval more accurate.